# Topic 3: Exactly-Once Semantics

> **Phase 6 → Week 11 → Topic 3**

---

## The Three Delivery Guarantees

```
At-most-once:    Message processed 0 or 1 times. Can be LOST.
  When: source offset committed before processing
  Risk: job crashes after commit but before write → data loss
  Use: when data loss is acceptable (metrics, logs)

At-least-once:   Message processed 1 or more times. Can be DUPLICATED.
  When: source offset committed after processing
  Risk: job crashes after write but before offset commit → duplicate
  Use: when duplicates can be deduplicated downstream

Exactly-once:    Message processed exactly 1 time. No loss, no duplicate.
  Requires: idempotent writes + atomic offset management
  Use: financial transactions, billing, inventory

Spark Structured Streaming provides exactly-once end-to-end
when using idempotent sinks (file/Delta) + checkpointing.
```

---

## How Spark Achieves Exactly-Once

```
Batch N processing:

  1. Read offsets from checkpoint
     offsets/N: {"orders": {"0": 1000, "1": 2000}}

  2. Fetch records from source (Kafka/files)

  3. Process and write to sink (idempotent)
     File sink: write to a temp path, then atomically rename
     Delta sink: use batch_id as idempotency key

  4. Commit to checkpoint
     commits/N: batch N done
     offsets/N+1: {"orders": {"0": 1100, "1": 2150}}

On restart after crash at step 3:
  - Spark sees commit/N is missing → batch N not done
  - Re-reads offsets from offsets/N (same source data)
  - Re-writes to sink — idempotent write → same result, no duplicate
  - Commits offsets/N+1

On restart after crash at step 4:
  - Spark sees commit/N exists but offsets/N+1 missing
  - Uses offsets/N+1 from the in-progress write → no reprocessing
```

---

## Interview Cheat Sheet

**Q: How does Spark Structured Streaming achieve exactly-once?**
> Three components work together: (1) **Replayable sources** — Kafka and file sources can replay data from a stored offset/checkpoint. (2) **Checkpointing** — Spark durably records the offsets it has read and the batches it has committed. (3) **Idempotent sinks** — file and Delta sinks produce the same result even if a batch is written twice (the second write is a no-op or overwrite). Together, these ensure that even on crash and restart, each message is processed exactly once in the output.

**Q: Is Kafka sink exactly-once?**
> The built-in Kafka sink is **at-least-once** — on retry, Spark may resend messages already sent to Kafka. True exactly-once to Kafka requires Kafka transactional producers (Kafka 0.11+) and enabling idempotent producer config. This is complex and rarely done in practice. A common workaround: write to Delta (exactly-once) and then a separate job reads Delta and writes to Kafka.

**Q: What is an idempotent write?**
> An idempotent write produces the same result whether executed once or multiple times. For file sinks: writing the same file twice overwrites the first (same data). For Delta: using `MERGE` with a batch ID as the match condition ensures the same batch is never applied twice. For JDBC: use `INSERT OR REPLACE` / `UPSERT` on a primary key.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time, os, shutil

spark = SparkSession.builder \
    .appName("Week11 - Exactly Once") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark ready")

## Part 1: Simulating Exactly-Once with foreachBatch + batch_id

In [ ]:
# Exactly-once pattern using foreachBatch + batch_id idempotency key
# The batch_id is monotonically increasing — same batch_id = same data on retry

IDEMPOTENT_OUT = "/tmp/idempotent_output"
CKPT = "/tmp/ckpt_idempotent"
for p in [IDEMPOTENT_OUT, CKPT]:
    if os.path.exists(p): shutil.rmtree(p)

written_batch_ids = set()  # track which batch_ids we've written

def idempotent_write(batch_df, batch_id):
    """
    Idempotent batch writer.
    If this batch_id was already written (e.g., retry), skip it.
    In production: check a 'processed_batches' table in your DB.
    """
    if batch_id in written_batch_ids:
        print(f"  Batch {batch_id}: SKIPPED (already written — idempotent!)")
        return

    count = batch_df.count()
    # Write with batch_id in the path for partitioning
    batch_df \
        .withColumn("batch_id", F.lit(batch_id)) \
        .write.mode("overwrite") \
        .parquet(f"{IDEMPOTENT_OUT}/batch_{batch_id}")

    written_batch_ids.add(batch_id)
    print(f"  Batch {batch_id}: wrote {count} rows to batch_{batch_id}/")

stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10) \
    .load() \
    .withColumn("amount", (F.col("value") % 100 + 1).cast("double"))

q = stream.writeStream \
    .foreachBatch(idempotent_write) \
    .option("checkpointLocation", CKPT) \
    .trigger(processingTime="3 seconds") \
    .start()

time.sleep(12)
q.stop()

print(f"\nWritten batch IDs: {sorted(written_batch_ids)}")
total = spark.read.parquet(IDEMPOTENT_OUT).count()
print(f"Total rows: {total}")

## Part 2: Checkpoint-Based Fault Tolerance Demo

In [ ]:
# Demonstrate: restart with checkpoint → no data loss, no duplicates
FILE_SRC = "/tmp/eo_file_source"
OUT_PATH = "/tmp/eo_output"
CKPT2   = "/tmp/ckpt_eo"

for p in [FILE_SRC, OUT_PATH, CKPT2]:
    if os.path.exists(p): shutil.rmtree(p)
    os.makedirs(p)

schema = StructType([
    StructField("event_id", StringType()),
    StructField("amount",   DoubleType()),
])

def write_files(batch_num, n=5):
    spark.createDataFrame(
        [(f"E{batch_num}{i:02d}", float(i * 10)) for i in range(n)],
        ["event_id", "amount"]
    ).coalesce(1).write.json(f"{FILE_SRC}/b{batch_num}")
    print(f"  Wrote batch {batch_num} ({n} events)")

# Run 1: write batches 1 and 2, process them
write_files(1)
write_files(2)

q1 = spark.readStream.schema(schema).json(FILE_SRC) \
    .withColumn("doubled", F.col("amount") * 2) \
    .writeStream \
    .format("parquet") \
    .option("path", OUT_PATH) \
    .option("checkpointLocation", CKPT2) \
    .trigger(once=True) \
    .start()

q1.awaitTermination()
run1_count = spark.read.parquet(OUT_PATH).count()
print(f"After run 1: {run1_count} rows")

# Run 2 (simulating restart): write new batch 3, restart with SAME checkpoint
write_files(3)

q2 = spark.readStream.schema(schema).json(FILE_SRC) \
    .withColumn("doubled", F.col("amount") * 2) \
    .writeStream \
    .format("parquet") \
    .option("path", OUT_PATH) \
    .option("checkpointLocation", CKPT2) \
    .trigger(once=True) \
    .start()

q2.awaitTermination()
run2_count = spark.read.parquet(OUT_PATH).count()
print(f"After run 2: {run2_count} rows (should be 15 = batch1+batch2+batch3)")
print("Batches 1 and 2 were NOT reprocessed — checkpoint tracked them")
print("No duplicates ✓ | No data loss ✓")

## Part 3: Delivery Semantics Reference

In [ ]:
print("""
Delivery Semantics by Source + Sink:
════════════════════════════════════════════════════════════════

Source                Sink            Guarantee
────────────────────  ──────────────  ────────────────────────
Kafka (offset-based)  File/Delta      Exactly-once ✓
Kafka                 Kafka           At-least-once (need txn Kafka for EO)
Kafka                 JDBC            At-least-once (need UPSERT for EO)
Kafka                 foreachBatch    Exactly-once if function is idempotent
File source           File/Delta      Exactly-once ✓
rate (test only)      any             Exactly-once ✓ (built-in)

Requirements for exactly-once:
  1. Source must be replayable (Kafka offsets, file checkpoint)
  2. Spark checkpoint is durable (HDFS/S3, not local disk)
  3. Sink is idempotent:
     - File sink: same batch → same file → overwrite (no dup)
     - Delta sink: transactional commit, replay = no-op
     - JDBC: use INSERT OR REPLACE / MERGE ON primary key
     - Kafka: need idempotent producer + transactional API

Making foreachBatch exactly-once:
  def write_batch(df, batch_id):
      # 1. Check if batch_id already processed (in a metadata table)
      already_done = check_metadata_table(batch_id)
      if already_done:
          return
      # 2. Write the data
      df.write.mode("append").parquet(path)
      # 3. Record batch_id as done (atomically with data write if possible)
      record_batch_id(batch_id)

  → If crash between step 2 and 3:
    On retry: batch_id not in metadata → rewrite (idempotent)
  → If crash after step 3:
    On retry: batch_id found → skip → no duplicate
════════════════════════════════════════════════════════════════
""")

## Exercises

1. Run the file source checkpoint demo above. After run 2, manually delete the checkpoint directory and run the query again. What happens? How many rows are in the output?
2. Write a `foreachBatch` function that writes to a JDBC SQLite table using `INSERT OR REPLACE` (upsert by `event_id`). Why does this give exactly-once semantics?
3. Explain the difference between: (a) data is not lost, and (b) data is exactly-once. Can you have (a) without (b)? Give an example.
4. Your streaming job writes to Kafka. A network timeout causes the job to retry batch 5. Some messages from batch 5 were already sent to Kafka before the timeout. What delivery guarantee do you have? What would you need to change to get exactly-once?
5. What is the minimum configuration needed to get exactly-once in Spark Structured Streaming? (List the three things you must set/use)